# Explore Butler regristry for LSSTCam

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-19

In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import lsst.afw.display as afwDisplay
import numpy as np
import pandas as pd
from astropy.time import Time
import pprint

In [ ]:
from lsst.daf.butler import Butler

In [ ]:
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.labelsize"] = "x-large"
plt.rcParams["axes.titlesize"] = "x-large"
plt.rcParams["xtick.labelsize"] = "x-large"
plt.rcParams["ytick.labelsize"] = "x-large"


In [ ]:
!eups list lsst_distrib

In [ ]:
repo = "main"
instrument = "LSSTCam"
date_start = 20250415
date_selection = 20250416
where_clause = "instrument = '" + f"{instrument}" + "'"
where_clause_date = where_clause + f"and day_obs >= {date_start}"
skymapName = "lsst_cells_v2"
COLLECTION_NAME = 'LSSTCam/raw/all' 

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "01_ExpButReg"
DIR_DATA_OUT = f"dataout_{NB_TAG}"  # input data written by notebook 03
DIR_FIGS_OUT = f"figsout_{NB_TAG}"  # output figures for this notebook
os.makedirs(DIR_FIGS_OUT, exist_ok=True)
os.makedirs(DIR_DATA_OUT, exist_ok=True)

In [ ]:
def savefig(name):
    """Save current figure as both PDF and PNG in DIR_FIGS_OUT."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS_OUT, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")
fignum =0

In [ ]:
os.getenv("LSST_RESOURCES_WEBDAV_CONFIG")

In [ ]:
os.getenv("DAF_BUTLER_REPOSITORY_INDEX")

## Butler init

In [ ]:
#!eups list -s

In [ ]:
butler = Butler(repo)
registry = butler.registry

In [ ]:
butler.registry.queryDatasetTypes()

In [ ]:
all_collections = registry.queryCollections()

print(f"Nombre total de collections trouvées : {len(all_collections)}")

# Afficher les 20 premières pour un aperçu
print("\nLes 20 premières collections :")
for i, collection_name in enumerate(all_collections):
    print(collection_name)
    if i >= 19:
        break

In [ ]:
# Essayez la même méthode, mais sur l'objet 'registry'
try:
    exposure_records = registry.queryDimensionRecords('exposure')
    
    # Le reste de votre code
    list_of_records = list(exposure_records)
    print(f"Nombre total d'expositions trouvées : {len(list_of_records)}")

except AttributeError as e:
    print(f"Erreur de l'API Gen 3 : {e}")
    print("Veuillez vérifier l'initialisation de votre Butler, notamment l'URI et les collections.")

In [ ]:
df_exposure = pd.DataFrame(
    {
        "id": pd.Series(dtype="int"),
        "obs_id": pd.Series(dtype="int"),
        "day_obs": pd.Series(dtype="int"),
        "seq_num": pd.Series(dtype="int"),
        "time_start": pd.Series(dtype="str"),  # ou 'datetime64[ns]' si c’est un datetime
        "time_end": pd.Series(dtype="str"),  # idem
        "type": pd.Series(dtype="str"),
        "target": pd.Series(dtype="str"),
        "filter": pd.Series(dtype="str"),
        "zenith_angle": pd.Series(dtype="float"),
        "expos": pd.Series(dtype="float"),  # ou 'int' selon le cas
        "ra": pd.Series(dtype="float"),
        "dec": pd.Series(dtype="float"),
        "skyangle": pd.Series(dtype="float"),
        "azimuth": pd.Series(dtype="float"),
        "zenith": pd.Series(dtype="float"),
        "science_program": pd.Series(dtype="str"),
        "jd": pd.Series(dtype="float"),
        "mjd": pd.Series(dtype="float"),
    }
)

In [ ]:
# save the data array in rows before saving in pandas dataframe
rows = []
for count, info in enumerate(registry.queryDimensionRecords("exposure", where=where_clause_date)):
    try:
        jd_start = info.timespan.begin.value
        jd_end = info.timespan.end.value
        the_Time_start = Time(jd_start, format="jd", scale="utc")
        the_Time_end = Time(jd_end, format="jd", scale="utc")
        mjd_start = the_Time_start.mjd
        mjd_end = the_Time_end.mjd
        isot_start = the_Time_start.isot
        isot_end = the_Time_end.isot

        if count == 0:
            print("===== Time Conversion Debug Info =====")
            print(f"JD start      : {jd_start} (type: {type(jd_start)})")
            print(f"JD end        : {jd_end} (type: {type(jd_end)})")
            print(f"MJD start     : {mjd_start} (type: {type(mjd_start)})")
            print(f"MJD end       : {mjd_end} (type: {type(mjd_end)})")
            print(f"ISOT start    : {isot_start} (type: {type(isot_start)})")
            print(f"ISOT end      : {isot_end} (type: {type(isot_end)})")
            print("=======================================")

        # put row in a dictionnary before stacking
        row = {
            "id": info.id,
            "obs_id": info.obs_id,
            "day_obs": info.day_obs,
            "seq_num": info.seq_num,
            "time_start": isot_start,
            "time_end": isot_end,
            "type": info.observation_type,
            "target": info.target_name,
            "filter": info.physical_filter,
            "zenith_angle": info.zenith_angle,
            "expos": info.exposure_time,  # Exemple : adapter selon ton objet
            "ra": info.tracking_ra,
            "dec": info.tracking_dec,
            "skyangle": info.sky_angle,
            "azimuth": info.azimuth,
            "zenith": info.zenith_angle,
            "science_program": info.science_program,
            "jd": float(jd_start),
            "mjd": float(mjd_start),
        }
        rows.append(row)

    except ValueError as e:
        print(f"Erreur de valeur : {e}")
    except FileNotFoundError as e:
        print(f"Fichier introuvable : {e}")
    except Exception as e:
        print(f"Erreur inattendue : {type(e).__name__} - {e}")
        print(f">>>   Unexpected error at row {count}:", sys.exc_info()[0])
        traceback.print_exc()  # affiche la stack trace complète


In [ ]:
# Création finale du DataFrame
df_exposure = pd.DataFrame(rows)

In [ ]:
df_exposure

## Check the information in these data

In [ ]:
list_of_columns = list(df_exposure.columns)
pprint.pprint(list_of_columns)

In [ ]:
list_of_types = list(df_exposure["type"].unique())
pprint.pprint(list_of_types)                             

## Select science types

In [ ]:
df_science = df_exposure[df_exposure.type == "science"]
df_science.reset_index(drop=True, inplace=True)

In [ ]:
df_science = df_science.copy()
df_science["band"] = df_science["filter"].apply(lambda x : x.split("_")[0])

In [ ]:
df_science

### Show targets in Science Exposures

In [ ]:
list_of_targets = list(df_science["target"].unique())
pprint.pprint(list_of_targets)

In [ ]:
TARGET_CATEGORIES = {
    "DDF": [
        r"ddf",
    ],
    "SCP": [
        r"\bscp\b",
    ],
    "LMC_SMC": [
        r"lmc_smc",
    ],
    "Euclid": [
        r"euclid_overlap",
    ],
    "Lowdust": [
        r"lowdust",
    ],
    "Dusty plane": [
        r"dusty_plane",
    ],
    "Bulge": [
        r"\bbulgy\b",
    ],
    "NES": [
        r"\bnes\b",
    ],
    "GW": [
        r"gw_case",
    ],
    "ToO": [
        r"\btoo_",
    ],
    "AOS": [
        r"\baos\b",
        r"straylight",
        r"mirror covers",
    ],
    "Standard stars": [
        r"\bhd\s",
        r"\bhip\s",
        r"\bsirius\b",
        r"\bcanopus\b",
        r"\balf cen\b",
        r"\balpha centauri\b",
    ],
    "Rubin SV": [
        r"rubin_sv",
    ],
}

In [ ]:

import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


TARGET_CATEGORIES = {
    "DDF": [r"ddf"],
    "SCP": [r"\bscp\b"],
    "LMC_SMC": [r"lmc_smc"],
    "Euclid": [r"euclid_overlap"],
    "Lowdust": [r"lowdust"],
    "Dusty plane": [r"dusty_plane"],
    "Bulge": [r"\bbulgy\b"],
    "NES": [r"\bnes\b"],
    "GW": [r"gw_case"],
    "ToO": [r"\btoo_"],
    "AOS": [r"\baos\b", r"straylight", r"mirror covers"],
    "Standard stars": [
        r"\bhd\s",
        r"\bhip\s",
        r"\bsirius\b",
        r"\bcanopus\b",
        r"\balf cen\b",
        r"\balpha centauri\b",
    ],
    "Rubin SV": [r"rubin_sv"],
}


def plot_target_distribution(
    df,
    column="target",
    categories=None,
    orientation="horizontal",
    min_count=1,
    other_label="Other",
    sort="category",
    figsize=None,
    top_n=None,
    title="Distribution of targets",
    show_counts=True,
    grid=True,
    ax=None,
    log_scale=False,
    log_base=10,
):
    """
    Plot the distribution of Rubin LSST targets.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.

    column : str
        Column containing target names.

    categories : dict or None
        Dictionary defining categories and regex patterns.
        Matching is case-insensitive.
        First matching category wins.

    orientation : {'horizontal', 'vertical'}
        Bar orientation.

    min_count : int
        Targets with fewer occurrences are grouped into Other.

    other_label : str
        Name of the residual group.

    sort : {'category', 'count', 'alphabetical'}
        Ordering of bars.

    figsize : tuple or None
        Figure size. Automatically determined if None.

    top_n : int or None
        Keep only the top N individual targets.
        Remaining targets are grouped into Other.

    title : str
        Plot title.

    show_counts : bool
        Display counts next to bars.

    grid : bool
        Display grid.

    ax : matplotlib.axes.Axes or None
        Existing axes.

    Returns
    -------
    fig, ax, counts
        Figure, axes and summary dataframe.
    """

    if orientation not in ("horizontal", "vertical"):
        raise ValueError("orientation must be 'horizontal' or 'vertical'")

    if sort not in ("category", "count", "alphabetical"):
        raise ValueError("Invalid sort option")

    if categories is None:
        categories = TARGET_CATEGORIES

    # --------------------------------------------------
    # 1. Count targets (case-insensitive normalization)
    # --------------------------------------------------

    targets = df[column].dropna().astype(str).str.strip()

    counts = (
        targets.value_counts()
        .rename_axis("target")
        .reset_index(name="count")
    )

    counts["normalized"] = counts["target"].str.casefold()

    # --------------------------------------------------
    # 2. Assign categories
    # --------------------------------------------------

    compiled_patterns = {
        category: [
            re.compile(pattern, flags=re.IGNORECASE)
            for pattern in patterns
        ]
        for category, patterns in categories.items()
    }

    def classify(target):

        for category, patterns in compiled_patterns.items():

            if any(pattern.search(target) for pattern in patterns):
                return category

        return "Unclassified"

    counts["category"] = counts["target"].apply(classify)

    # --------------------------------------------------
    # 3. Group rare targets
    # --------------------------------------------------

    if top_n is not None:
        keep = counts.nlargest(top_n, "count")["target"]
        counts.loc[~counts["target"].isin(keep), "target"] = other_label

    else:
        counts.loc[counts["count"] < min_count, "target"] = other_label

    # Combine targets grouped as Other
    counts = (
        counts.groupby(["target", "category"], as_index=False)["count"]
        .sum()
    )

    # --------------------------------------------------
    # 4. Define category ordering
    # --------------------------------------------------

    category_order = list(categories.keys()) + ["Unclassified"]

    if sort == "category":

        counts["category_order"] = counts["category"].map(
            {c: i for i, c in enumerate(category_order)}
        )

        counts = counts.sort_values(
            ["category_order", "count", "target"],
            ascending=[True, False, True]
        )

    elif sort == "count":

        counts = counts.sort_values("count", ascending=False)

    elif sort == "alphabetical":

        counts = counts.sort_values("target")

    # --------------------------------------------------
    # 5. Assign colors by category
    # --------------------------------------------------

    cmap = plt.get_cmap("tab20")

    color_map = {
        category: cmap(i % cmap.N)
        for i, category in enumerate(category_order + ["Other"])
    }

    # --------------------------------------------------
    # 6. Figure dimensions
    # --------------------------------------------------

    n = len(counts)

    if figsize is None:

        if orientation == "horizontal":
            figsize = (12, max(5, 0.32 * n))

        else:
            figsize = (max(12, 0.32 * n), 8)

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    colors = [
        color_map.get(cat, "grey")
        for cat in counts["category"]
    ]

    positions = np.arange(n)

    # --------------------------------------------------
    # 7. Plot
    # --------------------------------------------------

    if orientation == "horizontal":

        bars = ax.barh(
            positions,
            counts["count"],
            color=colors,
            edgecolor="black",
            linewidth=0.4,
        )

        ax.set_yticks(positions)
        ax.set_yticklabels(counts["target"])

        ax.invert_yaxis()

        ax.set_xlabel("Number of exposures")
        ax.set_ylabel("Target")

        if show_counts:
            ax.bar_label(bars, padding=3, fontsize=8)

        if grid:
            ax.xaxis.grid(True, alpha=0.3)
            ax.set_axisbelow(True)

    else:

        bars = ax.bar(
            positions,
            counts["count"],
            color=colors,
            edgecolor="black",
            linewidth=0.4,
        )

        ax.set_xticks(positions)
        ax.set_xticklabels(
            counts["target"],
            rotation=60,
            ha="right"
        )

        ax.set_ylabel("Number of exposures")
        ax.set_xlabel("Target")

        if show_counts:
            ax.bar_label(
                bars,
                padding=3,
                rotation=90 if n > 25 else 0,
                fontsize=8
            )

        if grid:
            ax.yaxis.grid(True, alpha=0.3)
            ax.set_axisbelow(True)


    if log_scale:
        if orientation == "horizontal":
            ax.set_xscale("log", base=log_base)
        else:
            ax.set_yscale("log", base=log_base)

    # --------------------------------------------------
    # 8. Legend
    # --------------------------------------------------

    present_categories = counts["category"].unique()

    handles = [
        mpatches.Patch(
            color=color_map.get(cat, "grey"),
            label=cat
        )
        for cat in category_order
        if cat in present_categories
    ]

    if handles:
        ax.legend(
            handles=handles,
            title="Target category",
            loc="best",
            fontsize=9
        )

    ax.set_title(title)

    fig.tight_layout()

    return fig, ax, counts

In [ ]:
fig, ax, counts = plot_target_distribution(
    df_science,
    orientation="horizontal",
    min_count=200,
    sort="category",
    log_scale=True,
)
fignum +=1
figfilename = f"Figure_{fignum:02d}_barh_distrib_targets"
savefig(figfilename)
plt.show()

In [ ]:
fig, ax, counts = plot_target_distribution(
    df_science,
    orientation="vertical",
    min_count=200,
    sort="category",
    log_scale=True,
)
fignum +=1
figfilename = f"Figure_{fignum:02d}_barv_distrib_targets"
savefig(figfilename)
plt.show()

In [ ]:
my_categories = {
    "Deep Drilling Fields": [r"ddf"],
    "Southern Celestial Pole": [r"\bscp\b"],
    "Magellanic Clouds": [r"lmc_smc"],
    "Milky Way": [
        r"lowdust",
        r"dusty_plane",
        r"\bbulgy\b",
    ],
    "Euclid": [r"euclid_overlap"],
}

fig, ax, counts = plot_target_distribution(
    df_science,
    categories=my_categories,
    orientation="vertical",
    min_count=200,
    log_scale=True,
)
fignum +=1
figfilename = f"Figure_{fignum:02d}_barv_distrib_targets"
savefig(figfilename)
plt.show()

### Show science_program

In [ ]:
list_of_science_program = list(df_science["science_program"].unique())
pprint.pprint(list_of_science_program)

## Save file

In [ ]:
def save_science_exposures(
    df,
    output_dir=".",
    prefix="Rubin_LSST_ScienceExposures",
    file_format="parquet",
):
    """
    Save science exposures with a filename containing
    the minimum and maximum day_obs.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.

    output_dir : str
        Output directory.

    prefix : str
        Filename prefix.

    file_format : str
        'parquet' or 'csv'.

    Returns
    -------
    filename : str
        Full output filename.
    """

    from pathlib import Path

    # Select science exposures
    df_science = df.loc[df["type"].str.lower() == "science"].copy()

    if df_science.empty:
        raise ValueError("No science exposures found.")

    # Determine day_obs range
    day_obs_min = df_science["day_obs"].min()
    day_obs_max = df_science["day_obs"].max()

    # Format dates
    day_obs_min = str(day_obs_min)
    day_obs_max = str(day_obs_max)

    # Create filename
    filename = (
        f"{prefix}_{day_obs_min}_{day_obs_max}.{file_format}"
    )

    filepath = Path(output_dir) / filename

    # Save dataframe
    if file_format == "parquet":
        df_science.to_parquet(filepath, index=False)

    elif file_format == "csv":
        df_science.to_csv(filepath, index=False)

    else:
        raise ValueError("file_format must be 'parquet' or 'csv'")

    print(f"Saved {len(df_science)} science exposures")
    print(f"day_obs range: {day_obs_min} - {day_obs_max}")
    print(f"File: {filepath}")

    return filepath

In [ ]:
filepath_parq = save_science_exposures(
    df_science,
    output_dir = DIR_DATA_OUT,
    file_format="parquet",
)
print(filepath_parq)

In [ ]:
if 0:
    filepath_csv = save_science_exposures(
        df_science,
        output_dir=DIR_DATA_OUT,
        file_format="csv",
    )
    print(filepath_csv)

## Plot

### Plot vs time

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


def plot_pernight_visits_per_band(
    df,
    bands=("u", "g", "r", "i", "z", "y"),
    figsize=(15, 6),
    date_rotation=45,
    show_legend=True,
):
    """
    Plot number of visits per night, per filter band, night by night.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing at least 'mjd', 'day_obs' and 'band'.
    bands : tuple
        Filter bands to display.
    figsize : tuple
        Figure size.
    date_rotation : float
        Rotation angle of the upper date axis labels.
    show_legend : bool
        Whether to display the legend.

    Returns
    -------
    fig, ax
        Matplotlib figure and primary axis.
    """

    # Work on a copy
    data = df.copy()

    # Keep valid science observations
    data = data.dropna(subset=["mjd", "day_obs", "band"])

    # Convert dates
    data["day_obs"] = pd.to_datetime(
        data["day_obs"].astype(str),
        format="%Y%m%d"
    )

    # Sort observations chronologically
    data = data.sort_values("mjd").reset_index(drop=True)

    # Compute cumulative number of visits per band and per night
    data["n_day"] = (
        data.groupby(["day_obs", "band"])
        .cumcount() + 1
    )

    # Define colors for LSST bands
    band_colors = {
        "u": "tab:purple",
        "g": "tab:blue",
        "r": "tab:green",
        "i": "tab:orange",
        "z": "tab:red",
        "y": "tab:brown",
    }

    fig, ax = plt.subplots(figsize=figsize)

    # Plot each band, night by night
    for band in bands:

        band_data = data[data["band"] == band]

        for night, night_data in band_data.groupby("day_obs"):

            ax.plot(
                night_data["mjd"],
                night_data["n_day"],
                color=band_colors.get(band, "gray"),
                linewidth=1.5,
                marker=".",
                markersize=3,
                label=band if night == band_data["day_obs"].iloc[0]
                else None,
            )

    # Primary x-axis: MJD
    ax.set_xlabel("Modified Julian Date (MJD)", fontsize=12)

    # Y-axis
    ax.set_ylabel("Cumulative number of visits per night", fontsize=12)

    ax.set_ylim(bottom=0)

    # Grid
    ax.grid(True, alpha=0.3, linestyle="--")

    # Secondary x-axis: calendar dates
    ax_top = ax.twiny()

    ax_top.set_xlim(ax.get_xlim())

    # Use the same MJD-to-date conversion for tick positions
    mjd_ticks = ax.get_xticks()
    date_ticks = pd.to_datetime(
        mjd_ticks + 2400000.5,
        unit="D",
        origin="julian"
    )

    ax_top.set_xticks(mjd_ticks)
    ax_top.set_xticklabels(
        date_ticks.strftime("%Y/%m/%d"),
        rotation=date_rotation,
        ha="left"
    )

    ax_top.set_xlabel("Observation date", fontsize=12, labelpad=10)

    # Legend
    if show_legend:
        handles, labels = ax.get_legend_handles_labels()

        # Remove duplicate labels
        unique = dict(zip(labels, handles))

        ax.legend(
            unique.values(),
            unique.keys(),
            title="Filter band",
            loc="upper left",
            frameon=True,
        )

    fig.tight_layout()

    return fig, ax


In [ ]:
fig, ax = plot_pernight_visits_per_band(df_science)
fignum +=1
figfilename = f"Figure_{fignum:02d}_hist_nb_visits_pernight_perband"
savefig(figfilename)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator


def plot_cumulative_visits_per_band(
    df,
    bands=("u", "g", "r", "i", "z", "y"),
    figsize=(15, 6),
    date_rotation=45,
):
    """
    Plot global cumulative visits per filter band,
    with one bin per observing night.

    Bottom x-axis: numerical MJD values.
    Top x-axis: calendar dates (YYYY/MM/DD).
    """

    # --------------------------------------------------
    # 1. Prepare data
    # --------------------------------------------------

    data = df.dropna(
        subset=["mjd", "day_obs", "band"]
    ).copy()

    data["day_obs"] = pd.to_datetime(
        data["day_obs"].astype(str),
        format="%Y%m%d"
    )

    # --------------------------------------------------
    # 2. Count visits per night and per band
    # --------------------------------------------------

    daily_counts = (
        data.groupby(["day_obs", "band"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=bands, fill_value=0)
        .sort_index()
    )

    all_nights = pd.date_range(
        daily_counts.index.min(),
        daily_counts.index.max(),
        freq="D"
    )

    daily_counts = daily_counts.reindex(
        all_nights,
        fill_value=0
    )

    # --------------------------------------------------
    # 3. Global cumulative sum
    # --------------------------------------------------

    cumulative = daily_counts.cumsum()

    # Convert observing dates to MJD
    mjd_nights = (
        cumulative.index - pd.Timestamp("1858-11-17")
    ).total_seconds() / 86400.0

    # --------------------------------------------------
    # 4. Colors
    # --------------------------------------------------

    band_colors = {
        "u": "tab:purple",
        "g": "tab:blue",
        "r": "tab:green",
        "i": "tab:orange",
        "z": "tab:red",
        "y": "tab:brown",
    }

    # --------------------------------------------------
    # 5. Plot
    # --------------------------------------------------

    fig, ax = plt.subplots(figsize=figsize)

    for band in bands:
        ax.step(
            mjd_nights,
            cumulative[band].values,
            where="post",
            color=band_colors.get(band, "gray"),
            linewidth=2,
            label=band,
        )

    # --------------------------------------------------
    # 6. Bottom axis: numerical MJD
    # --------------------------------------------------

    ax.set_xlabel("Modified Julian Date (MJD)", fontsize=12)
    ax.set_ylabel("Cumulative number of visits", fontsize=12)

    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3, linestyle="--")

    # Force numerical MJD ticks
    ax.xaxis.set_major_locator(MaxNLocator(nbins=10))
    ax.ticklabel_format(axis="x", style="plain", useOffset=False)

    # --------------------------------------------------
    # 7. Top axis: calendar dates
    # --------------------------------------------------

    ax_top = ax.twiny()

    # Synchronize limits
    ax_top.set_xlim(ax.get_xlim())

    # Use exactly the same tick positions
    mjd_ticks = ax.get_xticks()
    ax_top.set_xticks(mjd_ticks)

    # Convert MJD to calendar dates
    date_ticks = pd.to_datetime(
        mjd_ticks,
        unit="D",
        origin=pd.Timestamp("1858-11-17")
    )

    ax_top.set_xticklabels(
        date_ticks.strftime("%Y/%m/%d"),
        rotation=date_rotation,
        ha="left"
    )

    ax_top.set_xlabel("Observation date", fontsize=12, labelpad=10)

    # --------------------------------------------------
    # 8. Legend and layout
    # --------------------------------------------------

    ax.legend(
        title="Filter band",
        loc="upper left",
        frameon=True
    )

    fig.tight_layout()

    return fig, ax


In [ ]:
fig, ax = plot_cumulative_visits_per_band(df_science)
fignum +=1
figfilename = f"Figure_{fignum:02d}_cum_nb_visits_pernight_perband"
savefig(figfilename)
plt.show()